# 🌐 Privacy-Preserving ML Cloud Service\n## Ready-to-Use with Your ngrok Token\n\nThis notebook creates a real cloud ML service that:\n- ✅ Accepts only latent vectors (no raw data)\n- ✅ Trains multiple ML models\n- ✅ Provides REST API endpoints\n- ✅ Ensures privacy preservation\n\n## 🚀 Instructions:\n1. Run Cell 1: Install dependencies\n2. Run Cell 2: Setup ngrok (token already configured)\n3. Run Cell 3: Start the service\n4. Copy the ngrok URL that appears\n5. Use the URL in your local project\n\n**Your ngrok token is already configured!**

In [ ]:
# Cell 1: Install Dependencies\n!pip install flask pyngrok scikit-learn numpy pandas requests\nprint('✅ Dependencies installed!')

In [ ]:
# Cell 2: Setup ngrok with Your Token\n!ngrok authtoken 35iBqxHoSoRf6KNoTW35eFZp6oX_UVfyXEVXsKh4uyAVq6Zth\nprint('✅ ngrok configured with your token!')

In [ ]:
# Cell 3: Create Privacy-Preserving ML Server\nfrom flask import Flask, request, jsonify\nimport numpy as np\nimport json\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score\nfrom sklearn.model_selection import train_test_split\nimport time\nfrom datetime import datetime\nimport threading\nfrom pyngrok import ngrok\n\napp = Flask(__name__)\ntrained_models = {}\ntraining_results = {}\n\n@app.route('/', methods=['GET'])\ndef home():\n    return jsonify({\n        'service': 'Privacy-Preserving ML Cloud Service',\n        'status': 'running',\n        'deployment': 'google_colab',\n        'privacy_policy': 'Only latent vectors accepted - raw data automatically rejected',\n        'timestamp': datetime.now().isoformat()\n    })\n\n@app.route('/health', methods=['GET'])\ndef health():\n    return jsonify({\n        'status': 'healthy',\n        'service': 'Privacy-Preserving ML Cloud Service',\n        'deployment': 'google_colab',\n        'timestamp': datetime.now().isoformat(),\n        'privacy_preserved': True,\n        'raw_data_accepted': False\n    })\n\n@app.route('/train', methods=['POST'])\ndef train():\n    try:\n        print('\\n🚀 RECEIVED TRAINING REQUEST FROM CLIENT')\n        print('=' * 50)\n        \n        data = request.get_json()\n        \n        if not data or 'latent_vectors' not in data or 'labels' not in data:\n            return jsonify({'error': 'Missing latent_vectors or labels'}), 400\n        \n        latent_vectors = np.array(data['latent_vectors'])\n        labels = np.array(data['labels'])\n        metadata = data.get('metadata', {})\n        \n        print(f'📊 Received data:')\n        print(f'   Latent vectors shape: {latent_vectors.shape}')\n        print(f'   Labels shape: {labels.shape}')\n        print(f'   Privacy preserved: {metadata.get(\"privacy_preserved\", True)}')\n        print(f'   Raw data included: {metadata.get(\"original_data_included\", False)}')\n        \n        # PRIVACY CHECK - Reject raw data\n        if metadata.get('original_data_included', False):\n            print('❌ PRIVACY VIOLATION DETECTED - Raw data in payload!')\n            return jsonify({\n                'error': 'Privacy violation detected - raw data not allowed!',\n                'privacy_policy': 'This service only accepts latent vectors'\n            }), 400\n        \n        # Split data\n        try:\n            X_train, X_test, y_train, y_test = train_test_split(\n                latent_vectors, labels, test_size=0.2, random_state=42,\n                stratify=labels if len(np.unique(labels)) > 1 else None\n            )\n        except ValueError:\n            X_train, X_test, y_train, y_test = train_test_split(\n                latent_vectors, labels, test_size=0.2, random_state=42\n            )\n        \n        print(f'\\n🎯 Training models on Google Colab:')\n        print(f'   Training samples: {len(X_train)}')\n        print(f'   Test samples: {len(X_test)}')\n        print(f'   Feature dimensions: {latent_vectors.shape[1]}')\n        \n        # Train models\n        models = {\n            'logistic_regression': LogisticRegression(random_state=42, max_iter=1000),\n            'mlp_classifier': MLPClassifier(hidden_layer_sizes=(64, 32), random_state=42, max_iter=300),\n            'random_forest': RandomForestClassifier(n_estimators=50, random_state=42)\n        }\n        \n        results = {}\n        start_time = time.time()\n        \n        for name, model in models.items():\n            try:\n                print(f'\\n📈 Training {name.replace(\"_\", \" \").title()}...')\n                model_start = time.time()\n                \n                model.fit(X_train, y_train)\n                \n                train_pred = model.predict(X_train)\n                test_pred = model.predict(X_test)\n                \n                train_acc = accuracy_score(y_train, train_pred)\n                test_acc = accuracy_score(y_test, test_pred)\n                \n                model_time = time.time() - model_start\n                model_id = f'colab_{name}_{int(time.time())}'\n                \n                trained_models[model_id] = model\n                \n                results[name] = {\n                    'model_id': model_id,\n                    'train_accuracy': float(train_acc),\n                    'test_accuracy': float(test_acc),\n                    'training_time': float(model_time),\n                    'status': 'success'\n                }\n                \n                print(f'   ✅ {name.replace(\"_\", \" \").title()} Results:')\n                print(f'      Training Accuracy: {train_acc:.3f}')\n                print(f'      Test Accuracy: {test_acc:.3f}')\n                print(f'      Training Time: {model_time:.2f}s')\n                \n            except Exception as e:\n                print(f'   ❌ {name} training failed: {str(e)}')\n                results[name] = {\n                    'model_id': None,\n                    'error': str(e),\n                    'status': 'failed'\n                }\n        \n        training_time = time.time() - start_time\n        \n        # Find best model\n        successful = {k: v for k, v in results.items() if v['status'] == 'success'}\n        if successful:\n            best_model = max(successful.keys(), key=lambda k: successful[k]['test_accuracy'])\n            best_accuracy = successful[best_model]['test_accuracy']\n        else:\n            best_model = None\n            best_accuracy = 0.0\n        \n        response = {\n            'status': 'success',\n            'message': 'Cloud training completed successfully on Google Colab',\n            'deployment': 'google_colab',\n            'results': results,\n            'summary': {\n                'best_model': best_model,\n                'best_accuracy': float(best_accuracy),\n                'total_training_time': float(training_time),\n                'models_trained': len(successful),\n                'data_shape': latent_vectors.shape,\n                'privacy_preserved': True,\n                'raw_data_processed': False\n            },\n            'timestamp': datetime.now().isoformat()\n        }\n        \n        # Store results\n        session_id = datetime.now().isoformat()\n        training_results[session_id] = response\n        \n        print(f'\\n✅ GOOGLE COLAB TRAINING COMPLETED:')\n        print(f'   Best model: {best_model}')\n        print(f'   Best accuracy: {best_accuracy:.3f}')\n        print(f'   Total time: {training_time:.2f}s')\n        print(f'   Privacy preserved: ✅')\n        \n        return jsonify(response)\n        \n    except Exception as e:\n        error_response = {\n            'status': 'error',\n            'message': f'Training failed: {str(e)}',\n            'deployment': 'google_colab',\n            'timestamp': datetime.now().isoformat()\n        }\n        print(f'❌ Training error: {str(e)}')\n        return jsonify(error_response), 500\n\n# Start Flask server\ndef run_server():\n    app.run(host='0.0.0.0', port=5000, debug=False)\n\nserver_thread = threading.Thread(target=run_server)\nserver_thread.daemon = True\nserver_thread.start()\n\n# Create ngrok tunnel\npublic_url = ngrok.connect(5000)\n\nprint('\\n🌐 PRIVACY-PRESERVING ML CLOUD SERVICE STARTED!')\nprint('=' * 55)\nprint(f'🔗 Public URL: {public_url}')\nprint(f'📍 Local URL: http://localhost:5000')\nprint('\\n🛡️  Privacy Features:')\nprint('   ✅ Only latent vectors accepted')\nprint('   ❌ Raw data automatically rejected')\nprint('   🔒 HTTPS encryption enabled')\nprint('\\n📋 Available Endpoints:')\nprint(f'   GET  {public_url}/health - Health check')\nprint(f'   POST {public_url}/train - Train ML models')\nprint('\\n🚀 Ready for your privacy-preserving pipeline!')\nprint('\\n📋 COPY THIS URL FOR YOUR LOCAL PROJECT:')\nprint(f'\"{"{public_url}"}\"')\nprint('\\n⚠️  Keep this cell running to maintain the service!')

In [ ]:
# Cell 4: Keep Server Running\nimport time\n\ntry:\n    print('🔄 Server running... Press Ctrl+C to stop')\n    print('⚠️  Do not stop this cell - it keeps your service alive!')\n    print('\\n📊 Service Status:')\n    \n    while True:\n        print(f'   ✅ Service active at {datetime.now().strftime(\"%H:%M:%S\")}', end='\\r')\n        time.sleep(30)  # Update every 30 seconds\n        \nexcept KeyboardInterrupt:\n    ngrok.disconnect(public_url)\n    print('\\n🛑 Server stopped - Service is now offline')